<a href="https://colab.research.google.com/github/majumdarmanjari14-commits/PCB-defect-detector/blob/main/01to03rerun.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, shutil

PROJECT = '/content/drive/MyDrive/pcb-defect-detection'
DATA_DIR = os.path.join(PROJECT, 'data')

raw_path = os.path.join(DATA_DIR, 'DeepPCB')

if os.path.exists(raw_path):
    size_check = !du -sh {raw_path}
    print("Size before delete:", size_check)
    shutil.rmtree(raw_path)
    print("Deleted raw DeepPCB folder.")
else:
    print("Raw DeepPCB folder not found — may already be deleted, or DATA_DIR itself is missing.")

print("\nRemaining in DATA_DIR:")
print(os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "DATA_DIR does not exist")

Size before delete: ['84K\t/content/drive/MyDrive/pcb-defect-detection/data/DeepPCB']
Deleted raw DeepPCB folder.

Remaining in DATA_DIR:
[]


In [3]:
import os
import shutil
import pandas as pd
from PIL import Image

print("=" * 50)
print("STEP 1 — Setting up paths")
print("=" * 50)

PROJECT = '/content/drive/MyDrive/pcb-defect-detection'
DATA_DIR = os.path.join(PROJECT, 'data')
MODELS_DIR = os.path.join(PROJECT, 'models')
LOGS_DIR = os.path.join(PROJECT, 'logs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
print("Paths ready. DATA_DIR exists:", os.path.exists(DATA_DIR))

print("\n" + "=" * 50)
print("STEP 2 — Downloading DeepPCB dataset")
print("=" * 50)

dataset_path = os.path.join(DATA_DIR, 'DeepPCB')
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
    print("Removed old incomplete folder.")

os.makedirs(dataset_path, exist_ok=True)
exit_code = os.system(f"git clone https://github.com/tangsanli5201/DeepPCB.git {dataset_path}")
print("Git clone exit code:", exit_code, "(0 = success)")

pcb_data_path = os.path.join(DATA_DIR, 'DeepPCB', 'PCBData')
if not os.path.exists(pcb_data_path):
    raise RuntimeError("STOPPED: PCBData folder not found after clone. Download failed.")

board_folders = sorted([f for f in os.listdir(pcb_data_path)
                        if os.path.isdir(os.path.join(pcb_data_path, f))])

print(f"Groups found: {len(board_folders)}")
if len(board_folders) != 11:
    raise RuntimeError(f"STOPPED: Expected 11 groups, found {len(board_folders)}.")
print("All 11 groups confirmed present.")

print("\n" + "=" * 50)
print("STEP 3 — Building annotation table")
print("=" * 50)

records = []
for group in board_folders:
    group_path = os.path.join(pcb_data_path, group)
    board_id = group.replace('group', '')
    not_folder = os.path.join(group_path, f'{board_id}_not')
    img_folder = os.path.join(group_path, board_id)

    if not os.path.exists(not_folder):
        continue

    for txt_file in os.listdir(not_folder):
        if not txt_file.endswith('.txt'):
            continue
        txt_path = os.path.join(not_folder, txt_file)
        img_name = txt_file.replace('.txt', '_test.jpg')
        img_path = os.path.join(img_folder, img_name)
        if not os.path.exists(img_path):
            continue

        with open(txt_path, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]

        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                x1, y1, x2, y2, cls = (int(parts[0]), int(parts[1]),
                                        int(parts[2]), int(parts[3]), int(parts[4]))
                records.append({
                    'image_path': img_path,
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'label': cls, 'group': group
                })

df = pd.DataFrame(records)
print(f"Total annotation rows: {len(df)}")
if len(df) != 10013:
    raise RuntimeError(f"STOPPED: Expected 10013 rows, got {len(df)}.")
print("Row count confirmed correct.")

csv_path = os.path.join(DATA_DIR, 'defect_annotations.csv')
df.to_csv(csv_path, index=False)
print("Saved:", os.path.exists(csv_path), "| Size:", os.path.getsize(csv_path), "bytes")

print("\n" + "=" * 50)
print("STEP 4 — Cropping defect images")
print("=" * 50)

crops_dir = os.path.join(DATA_DIR, 'crops')
os.makedirs(crops_dir, exist_ok=True)
pad = 20
crop_records = []

for idx, row in df.iterrows():
    img = Image.open(row['image_path'])
    w, h = img.size
    x1p = max(0, row['x1'] - pad)
    y1p = max(0, row['y1'] - pad)
    x2p = min(w, row['x2'] + pad)
    y2p = min(h, row['y2'] + pad)
    crop = img.crop((x1p, y1p, x2p, y2p))

    crop_filename = f"{idx:05d}_label{row['label']}.jpg"
    crop_path = os.path.join(crops_dir, crop_filename)
    crop.save(crop_path)
    crop_records.append({'crop_path': crop_path, 'label': row['label']})

    if idx % 2000 == 0:
        print(f"  Cropped {idx}/{len(df)}")

crops_df = pd.DataFrame(crop_records)
print(f"Total crops saved: {len(crops_df)}")
if len(crops_df) != 10013:
    raise RuntimeError(f"STOPPED: Expected 10013 crops, got {len(crops_df)}.")

crops_csv_path = os.path.join(DATA_DIR, 'crops_labels.csv')
crops_df.to_csv(crops_csv_path, index=False)
print("Saved:", os.path.exists(crops_csv_path), "| Size:", os.path.getsize(crops_csv_path), "bytes")

print("\n" + "=" * 50)
print("ALL STEPS COMPLETE")
print("=" * 50)

STEP 1 — Setting up paths
Paths ready. DATA_DIR exists: True

STEP 2 — Downloading DeepPCB dataset
Git clone exit code: 0 (0 = success)
Groups found: 11
All 11 groups confirmed present.

STEP 3 — Building annotation table
Total annotation rows: 10013
Row count confirmed correct.
Saved: True | Size: 1288042 bytes

STEP 4 — Cropping defect images
  Cropped 0/10013
  Cropped 2000/10013
  Cropped 4000/10013
  Cropped 6000/10013
  Cropped 8000/10013
  Cropped 10000/10013
Total crops saved: 10013
Saved: True | Size: 740978 bytes

ALL STEPS COMPLETE
